In [ ]:
import os
import pandas as pd
import mlflow
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.neighbors import KNeighborsClassifier
from dotenv import load_dotenv

load_dotenv()

estimator = KNeighborsClassifier(n_neighbors=3)

os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.getenv("MLFLOW_S3_ENDPOINT_URL")
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

initial_csv_path = "s3://s3-student-mle-20260614-fb07c65e05/19/9829f204cdb4482598e011bf36c51d85/artifacts/models_artifacts/data/initial_data.csv"
pipeline_initial_path = "s3://s3-student-mle-20260614-fb07c65e05/19/9829f204cdb4482598e011bf36c51d85/artifacts/models"

storage_options = {
    "key": os.getenv("AWS_ACCESS_KEY_ID"),
    "secret": os.getenv("AWS_SECRET_ACCESS_KEY"),
    "client_kwargs": {"endpoint_url": "https://storage.yandexcloud.net"},
}
df_new: pd.DataFrame = pd.read_csv(
    initial_csv_path, storage_options=storage_options
)
pipeline = mlflow.sklearn.load_model(pipeline_initial_path)
preprocessor = pipeline.named_steps["preprocessor"]
model = pipeline.named_steps["model"]

X = df_new.drop(columns=["target", "end_date"], errors="ignore")
y = df_new["target"]
X_trans = preprocessor.transform(X)

model = CatBoostClassifier(auto_class_weights=auto_class_weights)

array([0, 0, 1, ..., 0, 1, 0])

In [15]:
from util.test_load_data_from_db import load_data_from_db

df = load_data_from_db()

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV

# загружаем датасет
iris = load_iris()
X = iris.data
y = iris.target

# разделяем датасет на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# определяем модель
model = LogisticRegression(solver="liblinear", multi_class="ovr")

# создаём сетку гиперпараметров для поиска
param_grid = {
    "C": [0.1, 1, 10, 100],  # сила регуляризации
    "penalty": ["l1", "l2"],  # тип регуляризации
}

# создаём объект GridSearchCV с кросс-валидацией на 5 фолдах, оптимизируя точность
grid_search = GridSearchCV(model, param_grid, cv=5, scoring="accuracy")

# выполняем поиск
grid_search.fit(X_train, y_train)

# выводим лучшие параметры и лучший счёт
print("Лучшие гиперпараметры:", grid_search.best_params_)
print("Лучший счет:", grid_search.best_score_)

# обучаем модель с лучшими параметрами на всём обучающем наборе
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

# оцениваем модель на тестовом наборе
test_score = best_model.score(X_test, y_test)
print("Точность на тестовой выборке:", test_score)

In [1]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import numpy as np

# загрузка данных
iris = load_iris()
X = iris.data
y = iris.target

# разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# создание модели
model = RandomForestClassifier()

# определение сетки гиперпараметров
param_distributions = {
    "n_estimators": [10, 50, 100, 200],  # количество деревьев в лесу
    "max_depth": [None, 10, 20, 30],  # максимальная глубина дерева
    "min_samples_split": [
        2,
        5,
        10,
    ],  # минимальное количество выборок, необходимое для разделения узла
    "min_samples_leaf": [
        1,
        2,
        4,
    ],  # минимальное количество выборок, необходимое на листовом узле
}

# создание объекта RandomizedSearchCV с 5 фолдами кросс-валидации (для примера)
random_search = RandomizedSearchCV(
    model, param_distributions, n_iter=100, cv=5, random_state=42, n_jobs=-1
)

# выполнение поиска
random_search.fit(X_train, y_train)

# выводим лучшие гиперпараметры и лучшую оценку
print("Лучшие гиперпараметры:", random_search.best_params_)
print("Лучший счёт:", random_search.best_score_)

# оценка модели на тестовой выборке
test_score = random_search.score(X_test, y_test)
print("Точность на тестовой выборке:", test_score)

Лучшие гиперпараметры: {'n_estimators': 50, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 10}
Лучший счёт: 0.9666666666666666
Точность на тестовой выборке: 1.0


In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
import os
import mlflow
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
)
from mlflow.models.signature import infer_signature

TABLE_NAME = "your_table_name"  # ваш код здесь

TRACKING_SERVER_HOST = "127.0.0.1"  # ваш код здесь
TRACKING_SERVER_PORT = 5003  # ваш код здесь

EXPERIMENT_NAME = "run_id_grid_search"  # ваш код здесь
RUN_NAME = "model_grid_search"  # ваш код здесь
REGISTRY_MODEL_NAME = "model_grid_search"  # ваш код здесь

features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = target  # ваш код здесь
stratify_column = target  # ваш код здесь
test_size = 0.2  # ваш код здесь

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features], df[target], test_size=test_size, random_state=42, stratify=y
)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")

loss_function = "Logloss"
task_type = "CPU"
random_seed = 0
iterations = 300
verbose = False

params = {
    # ваш код здесь (сетка гиперпараметров)
    "loss_function": ["Logloss", "CrossEntropy"],
    "task_type": ["CPU", "GPU"],
    "random_seed": [0, 42, 100],
    "iterations": [100, 200, 300],
    "verbose": [False, True],
}

model = CatBoostClassifier(
    loss_function=loss_function,
    task_type=task_type,
    random_seed=random_seed,
    iterations=iterations,
    verbose=verbose,
)

cv = GridSearchCV(
    estimator=model, param_grid=params, scoring="roc_auc", cv=2, n_jobs=-1
)

clf = cv.fit(X_train, y_train)  # ваш код здесь

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")


mlflow.set_tracking_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)
mlflow.set_registry_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)

cv_results = clf.cv_results_

best_params = clf.best_params_

loss_function = best_params.get("loss_function", loss_function)
task_type = best_params.get("task_type", task_type)
random_seed = best_params.get("random_seed", random_seed)
iterations = best_params.get("iterations", iterations)
verbose = best_params.get("verbose", verbose)

model_best = CatBoostClassifier(**best_params)

model_best.fit(X_train, y_train)

prediction = model_best.predict(X_test)
probas = model_best.predict_proba(X_test)[:, 1]

# расчёт метрик качества
metrics = {}

_, err1, _, err2 = confusion_matrix(
    y_test, prediction, normalize="all"
).ravel()
auc = roc_auc_score(y_test, probas)  # площадь под ROC-кривой
precision = precision_score(y_test, prediction)  # точность
recall = recall_score(y_test, prediction)  # полнота
f1 = f1_score(y_test, prediction)  # F1-мера
logloss = log_loss(y_test, prediction)  # LogLoss

# сохранение метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

# дополнительные метрики из результатов кросс-валидации
metrics["mean_fit_time"] = cv_results["mean_fit_time"].mean()
metrics["std_fit_time"] = cv_results["std_fit_time"].mean()
metrics["mean_test_score"] = cv_results["mean_test_score"].mean()
metrics["std_test_score"] = cv_results["std_test_score"].mean()
metrics["best_score"] = clf.best_score_

# настройки для логирования в MLFlow
pip_requirements = "requirements.txt"  # файл с зависимостями
signature = infer_signature(X_train, model_best.predict(X_train))
input_example = X_train.head(1)  # пример входных данных

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    mlflow.log_params(best_params)
    mlflow.log_metrics(metrics)
    cv_info = mlflow.sklearn.log_model(cv, artifact_path="cv")
    mlflow.catboost.log_model(
        cb_model=model_best,
        artifact_path="models",
        registered_model_name=REGISTRY_MODEL_NAME,
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
    )

Размер выборки для обучения: (5634, 3)
Размер выборки для теста: (1409, 3)
Learning rate set to 0.161862
Learning rate set to 0.161862
0:	learn: 0.6416589	total: 500us	remaining: 49.6ms
1:	learn: 0.5983984	total: 1.06ms	remaining: 52.2ms
2:	learn: 0.5692016	total: 1.46ms	remaining: 47.3ms
0:	learn: 0.6411900	total: 591us	remaining: 58.5ms
3:	learn: 0.5435823	total: 1.98ms	remaining: 47.6ms
1:	learn: 0.5978000	total: 1.08ms	remaining: 53.1ms
4:	learn: 0.5231967	total: 2.5ms	remaining: 47.6ms
Learning rate set to 0.161862
0:	learn: 0.6413210	total: 588us	remaining: 58.2ms
2:	learn: 0.5655993	total: 2.62ms	remaining: 84.8ms
Learning rate set to 0.161862
1:	learn: 0.5983161	total: 1.13ms	remaining: 55.3ms
5:	learn: 0.5076197	total: 4.13ms	remaining: 64.7ms
3:	learn: 0.5400605	total: 3.06ms	remaining: 73.4ms
4:	learn: 0.5194525	total: 3.51ms	remaining: 66.7ms
6:	learn: 0.4959238	total: 4.68ms	remaining: 62.2ms
2:	learn: 0.5662717	total: 1.8ms	remaining: 58.3ms
5:	learn: 0.5038874	total: 4.0

/Users/sergeyuser/Documents/Yandex Практикум/проекты/sprint2_part2_data/.venv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/sergeyuser/Documents/Yandex Практикум/проекты/sprint2_part2_data/.venv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/sergeyuser/Documents/Yandex Практикум/проекты/sprint2_part2_data/.venv/lib/python3.10/site-packages/catboost/core.py", line 5547, in fit
    self._fit(X, y, cat_fea

🏃 View run model_grid_search at: http://127.0.0.1:5003/#/experiments/20/runs/6a7d7b6227314b8bb065905af4796051
🧪 View experiment at: http://127.0.0.1:5003/#/experiments/20


In [28]:
from catboost import CatBoostClassifier
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
)
import os
import mlflow
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
)
from mlflow.models.signature import infer_signature

TABLE_NAME = "your_table_name"  # ваш код здесь

TRACKING_SERVER_HOST = "127.0.0.1"  # ваш код здесь
TRACKING_SERVER_PORT = 5003  # ваш код здесь

EXPERIMENT_NAME = "run_id_random_search"  # ваш код здесь
RUN_NAME = "model_grid_search"  # ваш код здесь
REGISTRY_MODEL_NAME = "model_grid_search"  # ваш код здесь

features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = target  # ваш код здесь
stratify_column = target  # ваш код здесь
test_size = 0.2  # ваш код здесь

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features], df[target], test_size=test_size, random_state=42, stratify=y
)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")

loss_function = "Logloss"
task_type = "CPU"
random_seed = 0
iterations = 300
verbose = False

params = {
    # ваш код здесь (сетка гиперпараметров)
    "loss_function": ["Logloss", "CrossEntropy"],
    "task_type": ["CPU", "GPU"],
    "random_seed": [0, 42, 100],
    "iterations": [100, 200, 300],
    "verbose": [False, True],
}

model = CatBoostClassifier(
    loss_function=loss_function,
    task_type=task_type,
    random_seed=random_seed,
    iterations=iterations,
    verbose=verbose,
)

cv = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    scoring="roc_auc",
    cv=2,
    n_jobs=-1,
)

clf = cv.fit(X_train, y_train)  # ваш код здесь

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")


mlflow.set_tracking_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)
mlflow.set_registry_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)

cv_results = clf.cv_results_

best_params = clf.best_params_

loss_function = best_params.get("loss_function", loss_function)
task_type = best_params.get("task_type", task_type)
random_seed = best_params.get("random_seed", random_seed)
iterations = best_params.get("iterations", iterations)
verbose = best_params.get("verbose", verbose)

model_best = CatBoostClassifier(**best_params)

model_best.fit(X_train, y_train)

prediction = model_best.predict(X_test)
probas = model_best.predict_proba(X_test)[:, 1]

# расчёт метрик качества
metrics = {}

_, err1, _, err2 = confusion_matrix(
    y_test, prediction, normalize="all"
).ravel()
auc = roc_auc_score(y_test, probas)  # площадь под ROC-кривой
precision = precision_score(y_test, prediction)  # точность
recall = recall_score(y_test, prediction)  # полнота
f1 = f1_score(y_test, prediction)  # F1-мера
logloss = log_loss(y_test, prediction)  # LogLoss

# сохранение метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

# дополнительные метрики из результатов кросс-валидации
metrics["mean_fit_time"] = cv_results["mean_fit_time"].mean()
metrics["std_fit_time"] = cv_results["std_fit_time"].mean()
metrics["mean_test_score"] = cv_results["mean_test_score"].mean()
metrics["std_test_score"] = cv_results["std_test_score"].mean()
metrics["best_score"] = clf.best_score_

# настройки для логирования в MLFlow
pip_requirements = "requirements.txt"  # файл с зависимостями
signature = infer_signature(X_train, model_best.predict(X_train))
input_example = X_train.head(1)  # пример входных данных

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    mlflow.log_params(best_params)
    mlflow.log_metrics(metrics)
    cv_info = mlflow.sklearn.log_model(cv, artifact_path="cv")
    mlflow.catboost.log_model(
        cb_model=model_best,
        artifact_path="models",
        registered_model_name=REGISTRY_MODEL_NAME,
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
    )

Размер выборки для обучения: (5634, 3)
Размер выборки для теста: (1409, 3)
Learning rate set to 0.070137
Learning rate set to 0.070137
0:	learn: 0.6631662	total: 56.1ms	remaining: 11.2s
1:	learn: 0.6302017	total: 56.7ms	remaining: 5.61s
2:	learn: 0.6014693	total: 57.3ms	remaining: 3.76s
0:	learn: 0.6637989	total: 56.3ms	remaining: 11.2s
3:	learn: 0.5794704	total: 58.1ms	remaining: 2.85s
1:	learn: 0.6312426	total: 57ms	remaining: 5.64s
4:	learn: 0.5592919	total: 58.8ms	remaining: 2.29s
5:	learn: 0.5437966	total: 59.3ms	remaining: 1.92s
2:	learn: 0.6066934	total: 57.6ms	remaining: 3.79s
3:	learn: 0.5846115	total: 58.3ms	remaining: 2.86s
6:	learn: 0.5301750	total: 60.4ms	remaining: 1.66s
4:	learn: 0.5621934	total: 58.9ms	remaining: 2.3s
7:	learn: 0.5183516	total: 61ms	remaining: 1.46s
5:	learn: 0.5451192	total: 60.1ms	remaining: 1.94s
8:	learn: 0.5078097	total: 61.9ms	remaining: 1.31s
6:	learn: 0.5305221	total: 60.8ms	remaining: 1.68s
9:	learn: 0.4994917	total: 63ms	remaining: 1.2s
7:	lea

/Users/sergeyuser/Documents/Yandex Практикум/проекты/sprint2_part2_data/.venv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
8 fits failed out of a total of 20.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
8 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/sergeyuser/Documents/Yandex Практикум/проекты/sprint2_part2_data/.venv/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/sergeyuser/Documents/Yandex Практикум/проекты/sprint2_part2_data/.venv/lib/python3.10/site-packages/catboost/core.py", line 5547, in fit
    self._fit(X, y, cat_features

🏃 View run model_grid_search at: http://127.0.0.1:5003/#/experiments/22/runs/c3ac1688b178444fb2f04c2d59427f81
🧪 View experiment at: http://127.0.0.1:5003/#/experiments/22


In [ ]:
TABLE_NAME = "clean_users_churn"  # ваш код здесь

TRACKING_SERVER_HOST = "127.0.0.1"  # ваш код здесь
TRACKING_SERVER_PORT = 5001  # ваш код здесь

EXPERIMENT_NAME = "model_random_search"  # ваш код здесь
RUN_NAME = "model_random_search"  # ваш код здесь
REGISTRY_MODEL_NAME = "model_grid_search"  # ваш код здесь

features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = "target"  # ваш код здесь
stratify_column = "target"  # ваш код здесь
test_size = 0.2  # ваш код здесь

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features], df[target], test_size=test_size, shuffle=False
)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")

loss_function = "Logloss"
task_type = "CPU"
random_seed = 0
iterations = 300
verbose = False

param_distributions = {
    # ваш код здесь (сетка гиперпараметров)
    "loss_function": ["Logloss", "CrossEntropy"],
    "task_type": ["CPU", "GPU"],
    "random_seed": [0, 42, 100],
    "iterations": [100, 200, 300],
    "verbose": [False, True],
}

model = CatBoostClassifier(
    loss_function=loss_function,
    task_type=task_type,
    random_seed=random_seed,
    iterations=iterations,
    verbose=verbose,
)

cv = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    scoring="roc_auc",
    cv=2,
    n_jobs=-1,
    n_iter=20,
)  # ваш код здесь

clf = cv.fit(X_train, y_train)  # ваш код здесь

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("S3_ACCESS_KEY")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("S3_SECRET_KEY")


mlflow.set_tracking_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)
mlflow.set_registry_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)

cv_results = pd.DataFrame(clf.cv_results_)

best_params = clf.best_params_  # ваш код здесь

model = CatBoostClassifier(
    **best_params,
    task_type=task_type,
    verbose=verbose,
    iterations=iterations,
    random_seed=random_seed,
    loss_function=loss_function,
)

model.fit(X_train, y_train)

prediction = model.predict(X_test)
probas = model.predict_proba(X_test)[:, 1]

# расчёт метрик качества
metrics = {}

_, err1, _, err2 = confusion_matrix(
    y_test, prediction, normalize="all"
).ravel()
auc = roc_auc_score(y_test, probas)  # площадь под ROC-кривой
precision = precision_score(y_test, prediction)  # точность
recall = recall_score(y_test, prediction)  # полнота
f1 = f1_score(y_test, prediction)  # F1-мера
logloss = log_loss(y_test, prediction)

# сохранение метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

#  дополнительные метрики из результатов кросс-валидации
metrics["mean_fit_time"] = cv_results["mean_fit_time"].mean()
metrics["std_fit_time"] = cv_results["std_fit_time"].mean()
metrics["mean_test_score"] = cv_results["mean_test_score"].mean()
metrics["std_test_score"] = cv_results["std_test_score"].mean()
metrics["best_score"] = clf.best_score_

# настройки для логирования в MLFlow
pip_requirements = "../requirements.txt"
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]

experiment_id = mlflow.get_experiment_by_name(
    EXPERIMENT_NAME
).experiment_id  # ваш код здесь

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    # ваш код здесь
    run_id = run.info.run_id

    mlflow.log_params(best_params)
    mlflow.log_metrics(metrics)
    cv_info = mlflow.sklearn.log_model(cv, artifact_path="cv")
    model_info = mlflow.catboost.log_model(
        cb_model=model_best,
        artifact_path="models",
        registered_model_name=REGISTRY_MODEL_NAME,
        signature=signature,
        input_example=input_example,
        pip_requirements=pip_requirements,
    )